# 📄 Research Paper Summarizer (RAG)

RAG-based summarizer for **Attention Is All You Need** and related LLM papers. Paper summaries are stored in **Chroma** for semantic retrieval. Uses **OpenRouter** via **LiteLLM** and **Gradio**.

**Setup:** Set `OPENROUTER_API_KEY` in `.env`. Embeddings use `sentence-transformers/all-MiniLM-L6-v2` (no API key).

## Install & Imports

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from litellm import completion
import gradio as gr
import chromadb
from chromadb.utils import embedding_functions

load_dotenv(override=True)

True

## Paper Knowledge Base

Summaries of key LLM papers the assistant can draw from when answering questions.

In [2]:
PAPER_SUMMARIES = {
    "attention_is_all_you_need": {
        "title": "Attention Is All You Need",
        "authors": "Vaswani et al. (2017)",
        "summary": "Introduces the Transformer architecture, which replaces recurrence and convolution with self-attention. Key contributions: (1) Multi-head self-attention allowing the model to attend to different positions and representations; (2) Position-wise feed-forward networks; (3) Positional encodings (sinusoidal) to inject sequence order; (4) Encoder-decoder structure with 6 layers each. Scaled dot-product attention: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V. Enables parallelization and achieves SOTA on WMT 2014 translation. Foundation for modern LLMs.",
        "key_concepts": ["self-attention", "multi-head attention", "positional encoding", "encoder-decoder", "transformer"]
    },
    "bert": {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers",
        "authors": "Devlin et al. (2018)",
        "summary": "Bidirectional encoder representation from Transformers. Pre-trains on massive text using Masked Language Modeling (MLM) and Next Sentence Prediction (NSP). Key innovation: bidirectional context (left and right) via MLM, unlike GPT's left-to-right. BERT-base: 110M params, 12 layers; BERT-large: 340M params. Achieves SOTA on GLUE, SQuAD. Fine-tuned for downstream NLP tasks. Uses WordPiece tokenization.",
        "key_concepts": ["bidirectional", "masked LM", "pre-training", "fine-tuning", "encoder-only"]
    },
    "gpt": {
        "title": "Language Models are Unsupervised Multitask Learners (GPT-2)",
        "authors": "Radford et al. (2019)",
        "summary": "GPT-2 extends GPT-1: decoder-only transformer, trained left-to-right. 1.5B parameters. Shows that scaling language models improves zero-shot task performance. No task-specific fine-tuning; learns tasks from text. GPT-3 (2020): 175B params, few-shot learning, in-context learning. Foundation for ChatGPT and modern autoregressive LLMs.",
        "key_concepts": ["decoder-only", "autoregressive", "zero-shot", "few-shot", "in-context learning"]
    },
    "llama": {
        "title": "LLaMA: Open and Efficient Foundation Language Models",
        "authors": "Touvron et al. (2023)",
        "summary": "Meta's open foundation models (7B–65B params). Key design: decoder-only, RMSNorm, SwiGLU activation, rotary positional embeddings (RoPE). Trained on public data only. Uses grouped-query attention (GQA) in larger models for efficiency. LLaMA 2 adds RLHF and longer context. LLaMA 3 (2024): 8B–70B, improved tokenizer, training data mix.",
        "key_concepts": ["RoPE", "SwiGLU", "grouped-query attention", "RLHF", "open weights"]
    },
    "rlhf": {
        "title": "Training Language Models to Follow Instructions with Human Feedback (InstructGPT)",
        "authors": "Ouyang et al. (2022)",
        "summary": "Aligns GPT with human preferences via RLHF (Reinforcement Learning from Human Feedback). Three stages: (1) Supervised fine-tuning on human-written demonstrations; (2) Train reward model on human comparisons; (3) Proximal Policy Optimization (PPO) to maximize reward. Produces more helpful, harmless, honest outputs. Foundation for ChatGPT.",
        "key_concepts": ["RLHF", "reward model", "PPO", "instruction tuning", "alignment"]
    },
    "mistral": {
        "title": "Mistral 7B",
        "authors": "Jiang et al. (2023)",
        "summary": "Efficient 7B model using sliding window attention (SWA) and grouped-query attention. SWA limits attention to a sliding window of recent tokens, reducing memory. Mistral-small and Mistral-large scale up. Strong performance at low cost. Mixture of Experts (MoE) variants for Mixtral.",
        "key_concepts": ["sliding window attention", "grouped-query attention", "MoE", "efficiency"]
    }
}

## Chroma Vector Database (RAG)

Setup Chroma, ingest paper summaries, and define `build_knowledge_context` for semantic retrieval.

In [ ]:
# Chroma Vector Database (RAG)
CHROMA_DB_PATH = Path("research_papers_chroma")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
collection = chroma_client.get_or_create_collection(
    name="research_papers",
    embedding_function=embedding_fn,
    metadata={"description": "LLM research paper summaries"},
)

# Ingest paper summaries into Chroma
def _paper_to_doc(key: str, paper: dict) -> tuple:
    text = f"{paper['title']} ({paper['authors']})\n\n{paper['summary']}\n\nKey concepts: {', '.join(paper['key_concepts'])}"
    meta = {"title": paper["title"], "authors": paper["authors"], "paper_key": key}
    return f"paper_{key}", meta, text

ids, metadatas, documents = [], [], []
for key, paper in PAPER_SUMMARIES.items():
    doc_id, meta, doc_text = _paper_to_doc(key, paper)
    ids.append(doc_id)
    metadatas.append(meta)
    documents.append(doc_text)
collection.upsert(ids=ids, metadatas=metadatas, documents=documents)
print(f"Ingested {len(documents)} papers into Chroma at {CHROMA_DB_PATH}")

# RAG retrieval: semantic search over Chroma
def build_knowledge_context(query: str, n_results: int = 4) -> str:
    """Retrieve relevant paper summaries via Chroma semantic search (RAG)."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas"],
    )
    docs = results["documents"][0] if results["documents"] and results["documents"][0] else []
    if not docs:
        return "No relevant papers found."
    return "\n\n".join(docs).strip()

## OpenRouter + LiteLLM Setup

In [4]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    print("⚠️ OPENROUTER_API_KEY not set. Add it to .env or your environment.")
else:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
    print(f"✓ OpenRouter key loaded (begins with {OPENROUTER_API_KEY[:10]}...)")

MODEL = "openrouter/openai/gpt-4o-mini"  # Fast and cost-effective; swap for others via OpenRouter

✓ OpenRouter key loaded (begins with sk-or-v1-a...)


## Chat Function

In [5]:
SYSTEM_PROMPT = """You are a helpful assistant that summarizes and explains research papers on LLMs and Transformers.
You have access to summaries of: Attention Is All You Need, BERT, GPT, LLaMA, InstructGPT/RLHF, Mistral.
Use the provided context to answer. Be concise but accurate. If asked about a paper not in context, say so.
Format answers clearly with bullet points or short paragraphs when helpful."""


def _to_content(msg):
    if isinstance(msg, str):
        return msg
    return getattr(msg, "content", str(msg) if msg else "")


def chat(message, history):
    if not OPENROUTER_API_KEY:
        return "Error: OPENROUTER_API_KEY not set. Add it to .env and restart."
    user_content = _to_content(message)
    if not user_content.strip():
        return "Please enter a question."
    context = build_knowledge_context(user_content)
    system_content = SYSTEM_PROMPT + "\n\n[Relevant paper summaries]\n\n" + context
    messages = [{"role": "system", "content": system_content}]
    for h in (history or []):
        role = h.get("role", "user")
        content = _to_content(h.get("content", ""))
        if content.strip():
            messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": user_content})
    try:
        response = completion(model=MODEL, messages=messages)
        return response.choices[0].message.content
    except Exception as e:
        return f"Error calling OpenRouter: {e}"

## Gradio Chat Interface

In [8]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="📄 Research Paper Summarizer",
    description="Ask about Attention Is All You Need, BERT, GPT, LLaMA, RLHF, Mistral, and related LLM papers.",
    examples=[
        "Summarize Attention Is All You Need",
        "What is multi-head attention?",
        "How does BERT differ from GPT?",
        "Explain RLHF and InstructGPT",
    ],
).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


this are the context parts [{'title': 'Attention Is All You Need', 'authors': 'Vaswani et al. (2017)', 'summary': 'Introduces the Transformer architecture, which replaces recurrence and convolution with self-attention. Key contributions: (1) Multi-head self-attention allowing the model to attend to different positions and representations; (2) Position-wise feed-forward networks; (3) Positional encodings (sinusoidal) to inject sequence order; (4) Encoder-decoder structure with 6 layers each. Scaled dot-product attention: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V. Enables parallelization and achieves SOTA on WMT 2014 translation. Foundation for modern LLMs.', 'key_concepts': ['self-attention', 'multi-head attention', 'positional encoding', 'encoder-decoder', 'transformer']}]
this are the context parts [{'title': 'Attention Is All You Need', 'authors': 'Vaswani et al. (2017)', 'summary': 'Introduces the Transformer architecture, which replaces recurrence and convolution with self-atte